Catastrophic cancellation. Compute the variance of [1000000.0, 1000001.0, 1000002.0] using the naive formula E[x^2] - E[x]^2 in float32. Then compute it using Welford's online algorithm. Compare the errors against the true variance (0.6667).

In [1]:
import numpy as np
data = [1000000.0, 1000001.0, 1000002.0]
n=len(data)
true_variance=np.var(np.array(data,dtype=np.float64))
print(f"True variance (float64): {true_variance:.10f}")
print(f"Expected: 0.6667")
print()

True variance (float64): 0.6666666667
Expected: 0.6667



- Compute E[x] and E[x^2]


In [2]:
data_f32 = np.array(data, dtype=np.float32)
mean_f32 = np.mean(data_f32)
mean_sq_f32 = np.mean(data_f32**2)
naive_variance = mean_sq_f32 - mean_f32**2
print("=== Naive Formula (float32) ===")
print(f"E[x]   = {mean_f32}")
print(f"E[x^2] = {mean_sq_f32}")
print(f"E[x]^2 = {mean_f32**2}")
print(f"Variance = E[x^2] - E[x]^2 = {naive_variance}")
print(f"Error: {abs(naive_variance - true_variance):.6f}")
print(f"Relative error: {abs(naive_variance - true_variance)/true_variance * 100:.2f}%")
print()

=== Naive Formula (float32) ===
E[x]   = 1000001.0
E[x^2] = 1000001961984.0
E[x]^2 = 1000002027520.0
Variance = E[x^2] - E[x]^2 = -65536.0
Error: 65536.666667
Relative error: 9830500.00%



- Method 2: Welford's online algorithm in float32

In [3]:
def welford_variance(data):
    n = 0
    mean = np.float32(0.0)
    M2 = np.float32(0.0)
    
    for x in data:
        x = np.float32(x)
        n += 1
        delta = x - mean
        mean += delta / np.float32(n)
        delta2 = x - mean
        M2 += delta * delta2
    
    if n < 2:
        return np.float32(0.0)
    else:
        return M2 / np.float32(n)

In [4]:
welford_var = welford_variance(data)

print("=== Welford's Online Algorithm (float32) ===")
print(f"Variance = {welford_var}")
print(f"Error: {abs(welford_var - true_variance):.6f}")
print(f"Relative error: {abs(welford_var - true_variance)/true_variance * 100:.6f}%")
print()

=== Welford's Online Algorithm (float32) ===
Variance = 0.6666666865348816
Error: 0.000000
Relative error: 0.000003%



In [5]:
# Compare catastrophic cancellation
print("=== Analysis ===")
print(f"Naive formula suffers from catastrophic cancellation:")
print(f"  E[x^2] = {mean_sq_f32} (~1e12)")
print(f"  E[x]^2 = {mean_f32**2} (~1e12)")
print(f"  Difference = {naive_variance} (should be ~0.667)")
print(f"  We're subtracting two large nearly-equal numbers,")
print(f"  losing ~6-7 significant digits in float32 (which has ~7 digits).")
print()
print(f"Welford's algorithm avoids this by working with deviations")
print(f"from the running mean, keeping numbers small and well-conditioned.")

=== Analysis ===
Naive formula suffers from catastrophic cancellation:
  E[x^2] = 1000001961984.0 (~1e12)
  E[x]^2 = 1000002027520.0 (~1e12)
  Difference = -65536.0 (should be ~0.667)
  We're subtracting two large nearly-equal numbers,
  losing ~6-7 significant digits in float32 (which has ~7 digits).

Welford's algorithm avoids this by working with deviations
from the running mean, keeping numbers small and well-conditioned.


Precision hunt. Find the smallest positive float32 value x such that 1.0 + x == 1.0 in Python. This is the machine epsilon. Verify it matches numpy.finfo(numpy.float32).eps

In [6]:
import numpy as np

# Method 1: Binary search / iterative approach to find smallest x where 1.0 + x == 1.0
def find_epsilon_iterative():
    """Find the smallest positive float32 x such that 1.0 + x == 1.0"""
    x = np.float32(1.0)
    
    # Keep halving until 1.0 + x == 1.0
    while np.float32(1.0) + x != np.float32(1.0):
        x = np.float32(x / 2)
    
    # Now x is the largest value where 1.0 + x != 1.0
    # The machine epsilon is the next representable value up (2*x)
    # Actually, let's be more careful:
    # We want the SMALLEST x such that 1.0 + x == 1.0
    # That's exactly the value we found (the first one that rounds to 1.0)
    return x

# Better approach: start from 1.0 and halve
def find_epsilon():
    """Find smallest positive float32 x such that 1.0 + x == 1.0"""
    x = np.float32(1.0)
    prev = x
    
    # Halve until 1.0 + x == 1.0
    while np.float32(1.0) + x != np.float32(1.0):
        prev = x
        x = np.float32(x / 2)
    
    # x is the smallest value where 1.0 + x == 1.0
    # prev is the largest value where 1.0 + prev != 1.0
    return x, prev

eps_found, largest_non_eps = find_epsilon()

print("=== Finding Machine Epsilon for float32 ===")
print(f"Smallest x where 1.0 + x == 1.0: {eps_found}")
print(f"Largest x where 1.0 + x != 1.0:  {largest_non_eps}")
print()

# Verify: the standard definition of machine epsilon is the difference
# between 1.0 and the next representable float
next_float = np.nextafter(np.float32(1.0), np.float32(2.0))
eps_nextafter = next_float - np.float32(1.0)

print(f"np.nextafter(1.0, 2.0) - 1.0 = {eps_nextafter}")
print()

# Compare with numpy.finfo
eps_numpy = np.finfo(np.float32).eps

print("=== Comparison ===")
print(f"Found epsilon (1.0 + x == 1.0):  {eps_found}  ({eps_found:.10e})")
print(f"nextafter-based epsilon:          {eps_nextafter}  ({eps_nextafter:.10e})")
print(f"numpy.finfo(float32).eps:         {eps_numpy}  ({eps_numpy:.10e})")
print()

print(f"Match (found == numpy):     {eps_found == eps_numpy}")
print(f"Match (nextafter == numpy): {eps_nextafter == eps_numpy}")
print()

# Let's verify the boundary conditions
print("=== Boundary Verification ===")
eps = eps_numpy
print(f"1.0 + eps         = {np.float32(1.0) + eps}  (should be != 1.0? No, should be > 1.0)")
print(f"1.0 + eps/2       = {np.float32(1.0) + np.float32(eps/2)}  (should be == 1.0)")
print(f"1.0 + eps*(1-δ)   = {np.float32(1.0) + np.float32(eps * np.float32(0.999))}  (rounds to 1.0)")
print()

# More precise check
print("=== Precise Boundary Check ===")
# Just below eps
just_below = np.nextafter(eps, np.float32(0))
print(f"eps                    = {eps:.10e}")
print(f"nextafter(eps, 0)      = {just_below:.10e}")
print(f"1.0 + eps              = {np.float32(1.0) + eps}")
print(f"1.0 + just_below       = {np.float32(1.0) + just_below}")
print(f"1.0 + just_below == 1.0? {np.float32(1.0) + just_below == np.float32(1.0)}")
print()

# Understanding: eps is the gap between 1.0 and the next float
print("=== Understanding the Result ===")
print(f"Representation of 1.0 in float32: exact")
print(f"Next float32 after 1.0: {next_float}")
print(f"Gap: {eps:.10e}")
print(f"This gap = 2^(-23) = {2.0**-23:.10e}")
print(f"Float32 has 23 mantissa bits, so eps = 2^-23 = {2.0**-23}")
print()
print(f"np.finfo(np.float32).eps = 2^-23? {eps == np.float32(2.0**-23)}")
print(f"np.finfo(np.float32).nmant = {np.finfo(np.float32).nmant}")

=== Finding Machine Epsilon for float32 ===
Smallest x where 1.0 + x == 1.0: 5.960464477539063e-08
Largest x where 1.0 + x != 1.0:  1.1920928955078125e-07

np.nextafter(1.0, 2.0) - 1.0 = 1.1920928955078125e-07

=== Comparison ===
Found epsilon (1.0 + x == 1.0):  5.960464477539063e-08  (5.9604644775e-08)
nextafter-based epsilon:          1.1920928955078125e-07  (1.1920928955e-07)
numpy.finfo(float32).eps:         1.1920928955078125e-07  (1.1920928955e-07)

Match (found == numpy):     False
Match (nextafter == numpy): True

=== Boundary Verification ===
1.0 + eps         = 1.0000001192092896  (should be != 1.0? No, should be > 1.0)
1.0 + eps/2       = 1.0  (should be == 1.0)
1.0 + eps*(1-δ)   = 1.0000001192092896  (rounds to 1.0)

=== Precise Boundary Check ===
eps                    = 1.1920928955e-07
nextafter(eps, 0)      = 1.1920928245e-07
1.0 + eps              = 1.0000001192092896
1.0 + just_below       = 1.0000001192092896
1.0 + just_below == 1.0? False

=== Understanding the Resu

Log-sum-exp edge cases. Test your logsumexp_stable function with: (a) all values equal, (b) one value much larger than the rest, (c) all values very negative (-1000). Verify it gives correct results where the naive version fails.

In [7]:
import math

def logsumexp_naive(values):
    return math.log(sum(math.exp(v) for v in values))

def logsumexp_stable(values):
    """Basic stable version — fails on empty input."""
    c = max(values)
    return c + math.log(sum(math.exp(v - c) for v in values))

def logsumexp_robust(values):
    """Robust version handling empty, all -inf, and NaN cases."""
    if len(values) == 0:
        return -math.inf                      # log(sum of empty set) = log(0) = -inf
    if any(math.isnan(v) for v in values):
        return math.nan
    c = max(values)
    if c == -math.inf:
        return -math.inf                      # all values are -inf
    if c == math.inf:
        return math.inf                       # any +inf dominates
    return c + math.log(sum(math.exp(v - c) for v in values))

In [8]:
import math

def check(name, values, expected):
    print(f"\n=== {name} ===")
    print(f"values   = {values}")
    print(f"expected = {expected:.10f}")

    try:
        naive = logsumexp_naive(values)
        print(f"naive    = {naive:.10f}   (err {abs(naive - expected):.3e})")
    except Exception as e:
        print(f"naive    = ERROR: {type(e).__name__}: {e}")

    try:
        stable = logsumexp_stable(values)
        print(f"stable   = {stable:.10f}   (err {abs(stable - expected):.3e})")
    except Exception as e:
        print(f"stable   = ERROR: {type(e).__name__}: {e}")

    try:
        robust = logsumexp_robust(values)
        print(f"robust   = {robust:.10f}   (err {abs(robust - expected):.3e})")
    except Exception as e:
        print(f"robust   = ERROR: {type(e).__name__}: {e}")

# (a) All values equal
check("(a) All equal", [5.0, 5.0, 5.0],
      math.log(3) + 5.0)                      # log(3·e^5) = 5 + log 3

# (b) One value much larger than the rest
check("(b) One dominant", [1000.0, 1.0, 2.0, 3.0],
      1000.0)                                 # e^1000 dominates; correction < 1e-400

# (c) All values very negative
check("(c) All very negative", [-1000.0, -1000.0, -1000.0],
      math.log(3) - 1000.0)                   # log(3·e^-1000)

# (d) Mixed large negatives (naive should give log(0) = -inf / ValueError)
check("(d) Mixed negatives", [-1000.0, -999.0, -1001.0],
      -999.0 + math.log(1 + 2*math.exp(-1)))  # -999 + log(1 + 2/e)

# (e) Empty input
check("(e) Empty", [], -math.inf)

# (f) All -inf
check("(f) All -inf", [-math.inf, -math.inf], -math.inf)

# (g) One +inf
check("(g) One +inf", [1.0, math.inf, 2.0], math.inf)

# (h) NaN present
check("(h) NaN present", [1.0, math.nan], math.nan)

# (i) Large positive (naive overflows to inf)
check("(i) Large positive", [1000.0, 1000.0], 1000.0 + math.log(2))


=== (a) All equal ===
values   = [5.0, 5.0, 5.0]
expected = 6.0986122887
naive    = 6.0986122887   (err 0.000e+00)
stable   = 6.0986122887   (err 0.000e+00)
robust   = 6.0986122887   (err 0.000e+00)

=== (b) One dominant ===
values   = [1000.0, 1.0, 2.0, 3.0]
expected = 1000.0000000000
naive    = ERROR: OverflowError: math range error
stable   = 1000.0000000000   (err 0.000e+00)
robust   = 1000.0000000000   (err 0.000e+00)

=== (c) All very negative ===
values   = [-1000.0, -1000.0, -1000.0]
expected = -998.9013877113
naive    = ERROR: ValueError: math domain error
stable   = -998.9013877113   (err 0.000e+00)
robust   = -998.9013877113   (err 0.000e+00)

=== (d) Mixed negatives ===
values   = [-1000.0, -999.0, -1001.0]
expected = -998.4485552861
naive    = ERROR: ValueError: math domain error
stable   = -998.5923940356   (err 1.438e-01)
robust   = -998.5923940356   (err 1.438e-01)

=== (e) Empty ===
values   = []
expected = -inf
naive    = ERROR: ValueError: math domain error
stable  

Gradient checking a neural network layer. Implement a single linear layer y = Wx + b and its analytical backward pass. Use numerical_gradient to verify correctness for a 3x2 weight matrix.

In [9]:
import numpy as np

np.random.seed(42)

class Linear:
    """
    Single linear layer: y = W @ x + b
    Shapes: W (out_features, in_features), x (in_features,), b (out_features,)
    """
    def __init__(self, in_features, out_features):
        # Small random init
        self.W = np.random.randn(out_features, in_features) * 0.1
        self.b = np.zeros(out_features)
        # Caches for backward
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return self.W @ x + self.b

    def backward(self, grad_output):
        """
        Given dL/dy (shape: out_features,), compute:
            dL/dW = grad_output ⊗ x   (outer product)
            dL/db = grad_output
            dL/dx = W^T @ grad_output
        """
        self.dW = np.outer(grad_output, self.x)
        self.db = grad_output.copy()
        grad_input = self.W.T @ grad_output
        return grad_input


def numerical_gradient(f, x, eps=1e-6):
    """Central-difference numerical gradient of scalar f w.r.t. array x."""
    x = x.astype(np.float64)
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        orig = x[idx]

        x[idx] = orig + eps
        f_plus = f(x)

        x[idx] = orig - eps
        f_minus = f(x)

        x[idx] = orig
        grad[idx] = (f_plus - f_minus) / (2 * eps)
        it.iternext()
    return grad


# ------------------------------------------------------------------
# Setup: 3x2 weight matrix  →  in_features=2, out_features=3
# ------------------------------------------------------------------
layer = Linear(in_features=2, out_features=3)

x = np.array([1.5, -2.0])            # input (2,)
y = layer.forward(x)                  # output (3,)
print("W shape:", layer.W.shape)
print("x       :", x)
print("y       :", y)
print()

# Fixed random upstream gradient (like dL/dy from the next layer)
rng = np.random.default_rng(0)
grad_output = rng.standard_normal(3)
print("grad_output (dL/dy):", grad_output)
print()

# ------------------------------------------------------------------
# Analytical gradients
# ------------------------------------------------------------------
layer.forward(x)                       # refresh cache
layer.backward(grad_output)
dW_analytical = layer.dW.copy()
db_analytical = layer.db.copy()
dx_analytical = layer.W.T @ grad_output

# ------------------------------------------------------------------
# Numerical gradients
# ------------------------------------------------------------------
# Scalar loss L = sum(grad_output * y)  →  dL/dy = grad_output
def loss_W(W_flat):
    W = W_flat.reshape(layer.W.shape)
    y = W @ x + layer.b
    return np.sum(grad_output * y)

def loss_b(b):
    y = layer.W @ x + b
    return np.sum(grad_output * y)

def loss_x(x_in):
    y = layer.W @ x_in + layer.b
    return np.sum(grad_output * y)

dW_numerical = numerical_gradient(loss_W, layer.W.copy().ravel()).reshape(layer.W.shape)
db_numerical = numerical_gradient(loss_b, layer.b.copy())
dx_numerical = numerical_gradient(loss_x, x.copy())

# ------------------------------------------------------------------
# Compare
# ------------------------------------------------------------------
def report(name, analytical, numerical):
    diff = np.abs(analytical - numerical)
    rel = diff / (np.abs(analytical) + np.abs(numerical) + 1e-12)
    print(f"--- {name} ---")
    print(f"analytical:\n{analytical}")
    print(f"numerical :\n{numerical}")
    print(f"abs diff  : max={diff.max():.3e}")
    print(f"rel diff  : max={rel.max():.3e}")
    print(f"OK? {np.allclose(analytical, numerical, rtol=1e-5, atol=1e-7)}")
    print()

report("dL/dW", dW_analytical, dW_numerical)
report("dL/db", db_analytical, db_numerical)
report("dL/dx", dx_analytical, dx_numerical)

W shape: (3, 2)
x       : [ 1.5 -2. ]
y       : [ 0.10215998 -0.20745269  0.01170439]

grad_output (dL/dy): [ 0.12573022 -0.13210486  0.64042265]

--- dL/dW ---
analytical:
[[ 0.18859533 -0.25146044]
 [-0.19815729  0.26420973]
 [ 0.96063398 -1.2808453 ]]
numerical :
[[ 0.18859533 -0.25146044]
 [-0.19815729  0.26420973]
 [ 0.96063398 -1.2808453 ]]
abs diff  : max=2.683e-12
rel diff  : max=5.334e-12
OK? True

--- dL/db ---
analytical:
[ 0.12573022 -0.13210486  0.64042265]
numerical :
[ 0.12573022 -0.13210486  0.64042265]
abs diff  : max=1.341e-12
rel diff  : max=5.334e-12
OK? True

--- dL/dx ---
analytical:
[-0.0173068  -0.03685303]
numerical :
[-0.0173068  -0.03685303]
abs diff  : max=3.498e-12
rel diff  : max=4.746e-11
OK? True

